In [33]:
import numpy as np 
import torch.nn.functional as F
import torch

Q = np.ones((10, 512)) * 1.0
K = np.ones((10, 512)) * 2.0
V = np.ones((10, 512)) * 3.0

def softmax(x):
    max_vals = np.max(x, axis=-1, keepdims=True)
    top = np.exp(x - max_vals) 
    bottom = np.sum(np.exp(x - max_vals), keepdims=True, axis=-1)
    return top / bottom

In [34]:
def softmax_causal(x):
    m_dim, n_dim = x.shape 
    assert len(x.shape) == 2 and "only work for dim2 for now"
    result = np.zeros(x.shape)
    for m in range(m_dim):
        max_val = np.max(x[m, 0:(m+1)])
        sum_val = np.sum(np.exp(x[m, 0:(m+1)] - np.array([max_val])))
        for n in range(n_dim):
            if n <= m: 
                result[m, n] = np.exp(x[m, n] - max_val) / sum_val 
            else: 
                result[m, n] = 0
    return result

def my_attention(Q, K, V): 
    dim_sequence, dim_model = Q.shape 
    _, dim_embedding = V.shape 
    output = np.zeros((dim_sequence, dim_embedding))
    # these come from the linalg.generic loop (i, j is inferred)
    for i in range(dim_sequence):
        for j in range(dim_embedding):
            # here emit the first linalg.matmul for the first attention
            pre_softmax = Q @ K.T
            # do the softmax calculation here, it it possible that you have emit to linalg, one to find max, and the other one to calculate the causal softmax
            softmax_calculation = softmax_causal(pre_softmax)
            # emit this loop through the linalg.generic, within the inner linalg.generic
            for k in range(dim_sequence):
                output[i, j] += softmax_calculation[i, k] * V[k, j]
            
    return output

my_atten = my_attention(Q, K, V)
torch_atten = F.scaled_dot_product_attention(torch.from_numpy(Q), torch.from_numpy(K), torch.from_numpy(V), is_causal=True, scale=1)
print(f"My Implementation:\n {my_atten}")
print(f"Pytorch Implementation:\n {torch_atten}")
print(f"All close: {torch.allclose(torch.from_numpy(my_atten), torch_atten)}")

My Implementation:
 [[3. 3. 3. ... 3. 3. 3.]
 [3. 3. 3. ... 3. 3. 3.]
 [3. 3. 3. ... 3. 3. 3.]
 ...
 [3. 3. 3. ... 3. 3. 3.]
 [3. 3. 3. ... 3. 3. 3.]
 [3. 3. 3. ... 3. 3. 3.]]
Pytorch Implementation:
 tensor([[3.0000, 3.0000, 3.0000,  ..., 3.0000, 3.0000, 3.0000],
        [3.0000, 3.0000, 3.0000,  ..., 3.0000, 3.0000, 3.0000],
        [3.0000, 3.0000, 3.0000,  ..., 3.0000, 3.0000, 3.0000],
        ...,
        [3.0000, 3.0000, 3.0000,  ..., 3.0000, 3.0000, 3.0000],
        [3.0000, 3.0000, 3.0000,  ..., 3.0000, 3.0000, 3.0000],
        [3.0000, 3.0000, 3.0000,  ..., 3.0000, 3.0000, 3.0000]],
       dtype=torch.float64)
All close: True


In [76]:
my_atten.shape

(10, 128)